# Project Mythos Standalone Kaggle Pipeline

Upload this `.ipynb` by itself. It embeds the current `src/mythos` package, writes it into `/kaggle/working/project_mythos_embedded/src`, then runs the plan-aligned ARC pipeline and writes `/kaggle/working/submission.json`.

Default mode is `pipeline` + `fallback`, which loads any configured model checkpoints and uses explicit fallback adapters for missing model stages.

## 1. Bootstrap Embedded Project Mythos Code

In [ ]:
from pathlib import Path
import os
import sys

EMBEDDED_FILES = {'src/mythos/__init__.py': '"""Project Mythos ARC testing harness."""\n\nfrom mythos.arc import ArcExample, ArcTask, ArcValidationError, load_challenges\nfrom mythos.submission import Prediction, TestPrediction\n\n__all__ = [\n    "ArcExample",\n    "ArcTask",\n    "ArcValidationError",\n    "Prediction",\n    "TestPrediction",\n    "load_challenges",\n]\n', 'src/mythos/__main__.py': '"""Top-level module help for `python -m mythos`."""\n\nfrom __future__ import annotations\n\n\ndef main() -> int:\n    print(\n        "Project Mythos commands:\\n"\n        "  python -m mythos.validate data/toy/challenges.json\\n"\n        "  python -m mythos.solve --solver fixture --challenges data/toy/challenges.json --out runs/submission.json\\n"\n        "  python -m mythos.score --pred runs/submission.json --solutions data/toy/solutions.json\\n"\n        "  python -m mythos.hrm_smoke --task data/toy/challenges.json"\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/arc.py': '"""ARC JSON loading and validation."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Optional, Tuple\n\nGrid = List[List[int]]\nSolutionMap = Dict[str, Tuple[Grid, ...]]\n\nMAX_GRID_ROWS = 30\nMAX_GRID_COLS = 30\nMIN_CELL_VALUE = 0\nMAX_CELL_VALUE = 9\n\n\nclass ArcValidationError(ValueError):\n    """Raised when ARC-style data is malformed."""\n\n\n@dataclass(frozen=True)\nclass ArcExample:\n    input: Grid\n    output: Optional[Grid] = None\n\n\n@dataclass(frozen=True)\nclass ArcTask:\n    id: str\n    train: Tuple[ArcExample, ...]\n    test: Tuple[ArcExample, ...]\n\n\ndef _json_load(path: str | Path) -> Any:\n    file_path = Path(path)\n    try:\n        with file_path.open("r", encoding="utf-8") as handle:\n            return json.load(handle)\n    except json.JSONDecodeError as exc:\n        raise ArcValidationError(f"{file_path} is not valid JSON: {exc}") from exc\n\n\ndef validate_grid(value: Any, *, field: str = "grid") -> Grid:\n    """Validate and copy an ARC grid."""\n\n    if not isinstance(value, list) or not value:\n        raise ArcValidationError(f"{field} must be a non-empty list of rows")\n    if len(value) > MAX_GRID_ROWS:\n        raise ArcValidationError(f"{field} has {len(value)} rows; max is {MAX_GRID_ROWS}")\n\n    rows: Grid = []\n    expected_width: int | None = None\n    for row_idx, row in enumerate(value):\n        if not isinstance(row, list) or not row:\n            raise ArcValidationError(f"{field}[{row_idx}] must be a non-empty list")\n        if expected_width is None:\n            expected_width = len(row)\n            if expected_width > MAX_GRID_COLS:\n                raise ArcValidationError(\n                    f"{field} has {expected_width} columns; max is {MAX_GRID_COLS}"\n                )\n        elif len(row) != expected_width:\n            raise ArcValidationError(\n                f"{field} must be rectangular; row 0 has {expected_width} columns "\n                f"but row {row_idx} has {len(row)}"\n            )\n\n        copied_row: List[int] = []\n        for col_idx, cell in enumerate(row):\n            if isinstance(cell, bool) or not isinstance(cell, int):\n                raise ArcValidationError(f"{field}[{row_idx}][{col_idx}] must be an integer")\n            if cell < MIN_CELL_VALUE or cell > MAX_CELL_VALUE:\n                raise ArcValidationError(\n                    f"{field}[{row_idx}][{col_idx}]={cell}; expected 0..9"\n                )\n            copied_row.append(cell)\n        rows.append(copied_row)\n    return rows\n\n\ndef _parse_example(raw: Any, *, task_id: str, split: str, index: int, require_output: bool) -> ArcExample:\n    if not isinstance(raw, Mapping):\n        raise ArcValidationError(f"{task_id}.{split}[{index}] must be an object")\n    if "input" not in raw:\n        raise ArcValidationError(f"{task_id}.{split}[{index}] is missing input")\n    output = raw.get("output")\n    if require_output and output is None:\n        raise ArcValidationError(f"{task_id}.{split}[{index}] is missing output")\n    return ArcExample(\n        input=validate_grid(raw["input"], field=f"{task_id}.{split}[{index}].input"),\n        output=validate_grid(output, field=f"{task_id}.{split}[{index}].output")\n        if output is not None\n        else None,\n    )\n\n\ndef _parse_examples(\n    raw_examples: Any, *, task_id: str, split: str, require_output: bool\n) -> Tuple[ArcExample, ...]:\n    if not isinstance(raw_examples, list) or not raw_examples:\n        raise ArcValidationError(f"{task_id}.{split} must be a non-empty list")\n    return tuple(\n        _parse_example(\n            raw_example,\n            task_id=task_id,\n            split=split,\n            index=index,\n            require_output=require_output,\n        )\n        for index, raw_example in enumerate(raw_examples)\n    )\n\n\ndef parse_task(task_id: str, raw_task: Any) -> ArcTask:\n    if not isinstance(raw_task, Mapping):\n        raise ArcValidationError(f"{task_id} must be an object")\n    if "train" not in raw_task or "test" not in raw_task:\n        raise ArcValidationError(f"{task_id} must contain train and test splits")\n    return ArcTask(\n        id=task_id,\n        train=_parse_examples(raw_task["train"], task_id=task_id, split="train", require_output=True),\n        test=_parse_examples(raw_task["test"], task_id=task_id, split="test", require_output=False),\n    )\n\n\ndef load_challenges(path: str | Path) -> Dict[str, ArcTask]:\n    """Load an ARC challenge JSON file keyed by task id."""\n\n    raw = _json_load(path)\n    if not isinstance(raw, Mapping) or not raw:\n        raise ArcValidationError("challenge file must be a non-empty object keyed by task id")\n    return {str(task_id): parse_task(str(task_id), raw_task) for task_id, raw_task in raw.items()}\n\n\ndef grid_shape(grid: Grid) -> Tuple[int, int]:\n    return len(grid), len(grid[0])\n\n\ndef grid_equal(left: Grid, right: Grid) -> bool:\n    return left == right\n\n\ndef copy_grid(grid: Grid) -> Grid:\n    return [row[:] for row in grid]\n\n\ndef _looks_like_grid(value: Any) -> bool:\n    try:\n        validate_grid(value)\n    except ArcValidationError:\n        return False\n    return True\n\n\ndef _solution_grids_from_value(task_id: str, value: Any) -> Tuple[Grid, ...]:\n    if isinstance(value, Mapping):\n        if "test" in value:\n            return tuple(\n                validate_grid(item["output"], field=f"{task_id}.test[{idx}].output")\n                for idx, item in enumerate(value["test"])\n                if isinstance(item, Mapping) and "output" in item\n            )\n        if "output" in value:\n            return (validate_grid(value["output"], field=f"{task_id}.output"),)\n    if _looks_like_grid(value):\n        return (validate_grid(value, field=f"{task_id}.output"),)\n    if isinstance(value, list):\n        grids: List[Grid] = []\n        for idx, item in enumerate(value):\n            if isinstance(item, Mapping) and "output" in item:\n                grids.append(validate_grid(item["output"], field=f"{task_id}[{idx}].output"))\n            elif _looks_like_grid(item):\n                grids.append(validate_grid(item, field=f"{task_id}[{idx}]"))\n            else:\n                raise ArcValidationError(f"{task_id}[{idx}] is not a solution grid")\n        if grids:\n            return tuple(grids)\n    raise ArcValidationError(f"{task_id} does not contain solution outputs")\n\n\ndef load_solutions(path: str | Path) -> SolutionMap:\n    raw = _json_load(path)\n    if not isinstance(raw, Mapping) or not raw:\n        raise ArcValidationError("solution file must be a non-empty object keyed by task id")\n    return {\n        str(task_id): _solution_grids_from_value(str(task_id), value)\n        for task_id, value in raw.items()\n    }\n\n\ndef attach_solutions(tasks: Mapping[str, ArcTask], solutions: SolutionMap) -> Dict[str, ArcTask]:\n    """Return tasks with test outputs filled from a solution map."""\n\n    attached: Dict[str, ArcTask] = {}\n    for task_id, task in tasks.items():\n        if task_id not in solutions:\n            raise ArcValidationError(f"missing solutions for task {task_id}")\n        if len(solutions[task_id]) != len(task.test):\n            raise ArcValidationError(\n                f"{task_id} has {len(task.test)} test items but "\n                f"{len(solutions[task_id])} solution outputs"\n            )\n        attached[task_id] = ArcTask(\n            id=task.id,\n            train=task.train,\n            test=tuple(\n                ArcExample(input=example.input, output=solutions[task_id][index])\n                for index, example in enumerate(task.test)\n            ),\n        )\n    return attached\n\n\ndef require_test_outputs(tasks: Iterable[ArcTask]) -> None:\n    for task in tasks:\n        for index, example in enumerate(task.test):\n            if example.output is None:\n                raise ArcValidationError(f"{task.id}.test[{index}] is missing output")\n', 'src/mythos/hrm_dataset.py': '"""Dataset-preparation glue for the external HRM checkout."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nfrom typing import Iterable\n\nfrom mythos.arc import ArcTask, ArcValidationError, require_test_outputs\n\n\ndef default_run_dir() -> Path:\n    import os\n\n    return Path(os.environ.get("MYTHOS_RUN_DIR", "runs"))\n\n\ndef prepare_hrm_raw_dataset(tasks: Iterable[ArcTask], output_dir: str | Path) -> Path:\n    """Write tasks into the directory shape HRM\'s ARC dataset builder expects."""\n\n    task_list = list(tasks)\n    require_test_outputs(task_list)\n\n    raw_data_dir = Path(output_dir)\n    eval_dir = raw_data_dir / "evaluation"\n    eval_dir.mkdir(parents=True, exist_ok=True)\n\n    for task in task_list:\n        raw_task = {\n            "train": [\n                {"input": example.input, "output": example.output}\n                for example in task.train\n            ],\n            "test": [\n                {"input": example.input, "output": example.output}\n                for example in task.test\n            ],\n        }\n        with (eval_dir / f"{task.id}.json").open("w", encoding="utf-8") as handle:\n            json.dump(raw_task, handle, indent=2)\n            handle.write("\\n")\n    return raw_data_dir\n\n\ndef build_hrm_dataset(\n    *,\n    hrm_repo_dir: str | Path,\n    raw_data_dir: str | Path,\n    output_dir: str | Path,\n    num_aug: int = 0,\n) -> subprocess.CompletedProcess[str]:\n    """Invoke HRM\'s own ARC dataset builder against a prepared raw-data directory."""\n\n    repo_dir = Path(hrm_repo_dir)\n    script = repo_dir / "dataset" / "build_arc_dataset.py"\n    if not script.exists():\n        raise ArcValidationError(f"HRM dataset builder not found: {script}")\n\n    output_path = Path(output_dir)\n    output_path.mkdir(parents=True, exist_ok=True)\n    command = [\n        sys.executable,\n        str(script),\n        "--dataset-dirs",\n        str(Path(raw_data_dir)),\n        "--output-dir",\n        str(output_path),\n        "--num-aug",\n        str(num_aug),\n    ]\n    return subprocess.run(\n        command,\n        cwd=repo_dir,\n        check=True,\n        capture_output=True,\n        text=True,\n    )\n', 'src/mythos/hrm_smoke.py': '"""CLI smoke test for the external HRM runtime."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom mythos.arc import ArcValidationError, attach_solutions, load_challenges, load_solutions\nfrom mythos.hrm_dataset import build_hrm_dataset, default_run_dir, prepare_hrm_raw_dataset\nfrom mythos.solvers.hrm import HRMEnvironment, HRMEnvironmentError\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Smoke-test external HRM integration.")\n    parser.add_argument("--task", required=True, help="ARC-style challenge JSON for the smoke run.")\n    parser.add_argument("--solutions", help="Optional solution JSON if --task omits test outputs.")\n    parser.add_argument("--run-dir", default=None, help="Output directory for smoke artifacts.")\n    parser.add_argument("--num-aug", type=int, default=0, help="HRM dataset-builder augmentation count.")\n    parser.add_argument(\n        "--skip-dataset-build",\n        action="store_true",\n        help="Only write the HRM raw-data layout; do not invoke HRM\'s dataset builder.",\n    )\n    args = parser.parse_args(argv)\n\n    started = time.perf_counter()\n    try:\n        tasks = load_challenges(args.task)\n        if args.solutions:\n            tasks = attach_solutions(tasks, load_solutions(args.solutions))\n\n        env = HRMEnvironment.from_env()\n        env.validate(require_cuda=True)\n        modules = env.import_modules()\n\n        torch = HRMEnvironment._import_torch()\n        torch.cuda.reset_peak_memory_stats()\n        checkpoint = env.load_checkpoint()\n\n        run_dir = Path(args.run_dir) if args.run_dir else default_run_dir() / "hrm_smoke"\n        raw_dir = prepare_hrm_raw_dataset(tasks.values(), run_dir / "raw" / "ARC-AGI-2" / "data")\n\n        dataset_build = None\n        if not args.skip_dataset_build:\n            result = build_hrm_dataset(\n                hrm_repo_dir=env.repo_dir,\n                raw_data_dir=raw_dir,\n                output_dir=run_dir / "data" / "arc-2-smoke",\n                num_aug=args.num_aug,\n            )\n            dataset_build = {\n                "returncode": result.returncode,\n                "stdout_tail": result.stdout[-2000:],\n                "stderr_tail": result.stderr[-2000:],\n            }\n    except (ArcValidationError, HRMEnvironmentError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    summary = {\n        "tasks": len(tasks),\n        "hrm_repo_dir": str(env.repo_dir),\n        "checkpoint_path": str(env.checkpoint_path),\n        "checkpoint_type": type(checkpoint).__name__,\n        "imported_modules": sorted(modules),\n        "raw_data_dir": str(raw_dir),\n        "dataset_build": dataset_build,\n        "elapsed_seconds": round(time.perf_counter() - started, 3),\n        "cuda_peak_memory_bytes": int(torch.cuda.max_memory_allocated()),\n    }\n    print(json.dumps(summary, indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/kaggle_models.py': '"""Kaggle model input auto-discovery.\n\nKaggle submissions usually mount model code and checkpoints under\n`/kaggle/input/<dataset-name>/...`. This module scans those inputs and sets the\nenvironment variables consumed by `mythos.models.ModelRegistry`.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport os\nimport subprocess\nimport urllib.request\nfrom typing import Iterable\n\n\nCHECKPOINT_SUFFIXES = (".pt", ".pth", ".ckpt", ".bin")\n\n\nMODEL_ENV_KEYS = (\n    "IJEPA_REPO_DIR",\n    "IJEPA_CHECKPOINT_PATH",\n    "HRM_TEXT_REPO_DIR",\n    "HRM_TEXT_CHECKPOINT_PATH",\n    "WORLD_MODEL_CHECKPOINT_PATH",\n    "TTT_LORA_CHECKPOINT_PATH",\n    "HRM_REPO_DIR",\n    "HRM_CHECKPOINT_PATH",\n)\n\nHF_MODEL_SPECS = (\n    {\n        "name": "jepa",\n        "repo_id_env": "IJEPA_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "IJEPA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_text",\n        "repo_id_env": "HRM_TEXT_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "HRM_TEXT_CHECKPOINT_PATH",\n    },\n    {\n        "name": "world_model",\n        "repo_id_env": "WORLD_MODEL_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "WORLD_MODEL_CHECKPOINT_PATH",\n    },\n    {\n        "name": "ttt_lora",\n        "repo_id_env": "TTT_LORA_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "TTT_LORA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_l_module",\n        "repo_id_env": "HRM_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "HRM_CHECKPOINT_PATH",\n    },\n)\n\nGIT_REPO_SPECS = (\n    {\n        "name": "jepa",\n        "url_env": "IJEPA_GIT_REPO_URL",\n        "repo_dir_env": "IJEPA_REPO_DIR",\n    },\n    {\n        "name": "hrm_text",\n        "url_env": "HRM_TEXT_GIT_REPO_URL",\n        "repo_dir_env": "HRM_TEXT_REPO_DIR",\n    },\n    {\n        "name": "hrm_l_module",\n        "url_env": "HRM_GIT_REPO_URL",\n        "repo_dir_env": "HRM_REPO_DIR",\n    },\n)\n\nDIRECT_CHECKPOINT_SPECS = (\n    {\n        "name": "jepa",\n        "url_env": "IJEPA_CHECKPOINT_URL",\n        "checkpoint_env": "IJEPA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_text",\n        "url_env": "HRM_TEXT_CHECKPOINT_URL",\n        "checkpoint_env": "HRM_TEXT_CHECKPOINT_PATH",\n    },\n    {\n        "name": "world_model",\n        "url_env": "WORLD_MODEL_CHECKPOINT_URL",\n        "checkpoint_env": "WORLD_MODEL_CHECKPOINT_PATH",\n    },\n    {\n        "name": "ttt_lora",\n        "url_env": "TTT_LORA_CHECKPOINT_URL",\n        "checkpoint_env": "TTT_LORA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_l_module",\n        "url_env": "HRM_CHECKPOINT_URL",\n        "checkpoint_env": "HRM_CHECKPOINT_PATH",\n    },\n)\n\n\ndef download_git_code_repositories(\n    output_root: str | Path = "/kaggle/working/model_code",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Clone configured Git repos for model code and export repo env vars."""\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "cloned": {},\n        "skipped": [],\n        "errors": {},\n    }\n    configured = [spec for spec in GIT_REPO_SPECS if os.environ.get(str(spec["url_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["url_env"]) for spec in GIT_REPO_SPECS]\n        return result\n\n    root = Path(output_root)\n    root.mkdir(parents=True, exist_ok=True)\n    for spec in configured:\n        name = str(spec["name"])\n        url = os.environ[str(spec["url_env"])]\n        repo_dir = root / name\n        try:\n            if not repo_dir.exists():\n                subprocess.run(\n                    ["git", "clone", "--depth", "1", url, str(repo_dir)],\n                    check=True,\n                    capture_output=True,\n                    text=True,\n                )\n            if apply:\n                os.environ.setdefault(str(spec["repo_dir_env"]), str(repo_dir))\n            result["cloned"][name] = {\n                "url": url,\n                "repo_dir": str(repo_dir),\n                "repo_dir_env": spec["repo_dir_env"],\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n    return result\n\n\ndef download_direct_checkpoint_inputs(\n    output_root: str | Path = "/kaggle/working/model_inputs/direct",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Download configured direct checkpoint URLs and export checkpoint env vars."""\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "downloaded": {},\n        "skipped": [],\n        "errors": {},\n    }\n    configured = [spec for spec in DIRECT_CHECKPOINT_SPECS if os.environ.get(str(spec["url_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["url_env"]) for spec in DIRECT_CHECKPOINT_SPECS]\n        return result\n\n    root = Path(output_root)\n    root.mkdir(parents=True, exist_ok=True)\n    for spec in configured:\n        name = str(spec["name"])\n        url = os.environ[str(spec["url_env"])]\n        target = root / name / _filename_from_url(url)\n        try:\n            target.parent.mkdir(parents=True, exist_ok=True)\n            if not target.exists():\n                urllib.request.urlretrieve(url, target)\n            checkpoint_env = str(spec["checkpoint_env"])\n            if apply:\n                os.environ.setdefault(checkpoint_env, str(target))\n            result["downloaded"][name] = {\n                "url": url,\n                "checkpoint": str(target),\n                "checkpoint_env": checkpoint_env,\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n    return result\n\n\ndef download_huggingface_model_inputs(\n    output_root: str | Path = "/kaggle/working/model_inputs",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Download configured Hugging Face model repos before model loading.\n\n    Configure with env vars such as `HRM_HF_REPO_ID`. Optional env vars named\n    `<PREFIX>_HF_CHECKPOINT_GLOB` can narrow checkpoint selection, for example\n    `HRM_HF_CHECKPOINT_GLOB="*.pt"`.\n    """\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "downloaded": {},\n        "skipped": [],\n        "errors": {},\n    }\n\n    configured = [spec for spec in HF_MODEL_SPECS if os.environ.get(str(spec["repo_id_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["repo_id_env"]) for spec in HF_MODEL_SPECS]\n        return result\n\n    try:\n        from huggingface_hub import snapshot_download\n    except Exception as exc:\n        result["errors"]["huggingface_hub"] = (\n            "huggingface_hub is not installed or cannot be imported: " + str(exc)\n        )\n        return result\n\n    output_dir = Path(output_root)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    for spec in configured:\n        name = str(spec["name"])\n        repo_id_env = str(spec["repo_id_env"])\n        repo_id = os.environ[repo_id_env]\n        local_dir = output_dir / name\n        glob_env = f"{repo_id_env.removesuffix(\'_REPO_ID\')}_CHECKPOINT_GLOB"\n        checkpoint_glob = os.environ.get(glob_env)\n\n        try:\n            downloaded = Path(\n                snapshot_download(\n                    repo_id=repo_id,\n                    local_dir=local_dir,\n                    local_dir_use_symlinks=False,\n                )\n            )\n            checkpoint = _find_checkpoint_by_glob(downloaded, checkpoint_glob)\n            if checkpoint is None:\n                result["errors"][name] = (\n                    f"downloaded {repo_id} to {downloaded}, but found no checkpoint "\n                    f"matching {checkpoint_glob or CHECKPOINT_SUFFIXES}"\n                )\n                continue\n\n            repo_dir_env = spec["repo_dir_env"]\n            checkpoint_env = str(spec["checkpoint_env"])\n            if apply:\n                if repo_dir_env is not None:\n                    os.environ.setdefault(str(repo_dir_env), str(downloaded))\n                os.environ.setdefault(checkpoint_env, str(checkpoint))\n\n            result["downloaded"][name] = {\n                "repo_id": repo_id,\n                "local_dir": str(downloaded),\n                "checkpoint": str(checkpoint),\n                "repo_dir_env": repo_dir_env,\n                "checkpoint_env": checkpoint_env,\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n\n    return result\n\n\ndef autodiscover_model_inputs(\n    input_root: str | Path = "/kaggle/input",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Find likely model repos/checkpoints and optionally export env vars."""\n\n    root = Path(input_root)\n    result: dict[str, object] = {\n        "input_root": str(root),\n        "exists": root.exists(),\n        "set": {},\n        "missing": [],\n    }\n    if not root.exists():\n        result["missing"] = list(MODEL_ENV_KEYS)\n        return result\n\n    discovered: dict[str, Path] = {}\n\n    hrm_repo = _find_hrm_repo(root)\n    hrm_checkpoint = _find_checkpoint(root, include=("hrm",), exclude=("text", "lora", "adapter"))\n    if hrm_repo is not None and hrm_checkpoint is not None:\n        discovered["HRM_REPO_DIR"] = hrm_repo\n        discovered["HRM_CHECKPOINT_PATH"] = hrm_checkpoint\n\n    ijepa_repo = _find_named_repo(root, names=("ijepa", "i-jepa", "jepa"))\n    ijepa_checkpoint = _find_checkpoint(root, include=("ijepa", "i-jepa", "jepa"))\n    if ijepa_repo is not None and ijepa_checkpoint is not None:\n        discovered["IJEPA_REPO_DIR"] = ijepa_repo\n        discovered["IJEPA_CHECKPOINT_PATH"] = ijepa_checkpoint\n\n    hrm_text_repo = _find_named_repo(root, names=("hrm-text", "hrm_text", "hrmtext"))\n    hrm_text_checkpoint = _find_checkpoint(root, include=("hrm", "text"), require_all=True)\n    if hrm_text_repo is not None and hrm_text_checkpoint is not None:\n        discovered["HRM_TEXT_REPO_DIR"] = hrm_text_repo\n        discovered["HRM_TEXT_CHECKPOINT_PATH"] = hrm_text_checkpoint\n\n    world_model_checkpoint = _find_checkpoint(root, include=("world", "transition"))\n    if world_model_checkpoint is not None:\n        discovered["WORLD_MODEL_CHECKPOINT_PATH"] = world_model_checkpoint\n\n    lora_checkpoint = _find_checkpoint(root, include=("lora", "adapter"))\n    if lora_checkpoint is not None:\n        discovered["TTT_LORA_CHECKPOINT_PATH"] = lora_checkpoint\n\n    set_values: dict[str, str] = {}\n    for key, path in discovered.items():\n        if key in os.environ:\n            set_values[key] = os.environ[key]\n            continue\n        if apply:\n            os.environ[key] = str(path)\n        set_values[key] = str(path)\n\n    result["set"] = set_values\n    result["missing"] = [key for key in MODEL_ENV_KEYS if key not in set_values and key not in os.environ]\n    return result\n\n\ndef _find_hrm_repo(root: Path) -> Path | None:\n    for path in _iter_dirs(root):\n        if (path / "evaluate.py").exists() and (path / "dataset" / "build_arc_dataset.py").exists():\n            return path\n    return None\n\n\ndef _find_named_repo(root: Path, *, names: Iterable[str]) -> Path | None:\n    lowered_names = tuple(name.lower() for name in names)\n    for path in _iter_dirs(root):\n        path_text = path.as_posix().lower()\n        if any(name in path_text for name in lowered_names):\n            return path\n    return None\n\n\ndef _find_checkpoint(\n    root: Path,\n    *,\n    include: Iterable[str],\n    exclude: Iterable[str] = (),\n    require_all: bool = False,\n) -> Path | None:\n    include_terms = tuple(term.lower() for term in include)\n    exclude_terms = tuple(term.lower() for term in exclude)\n    candidates = []\n    for path in root.rglob("*"):\n        if not path.is_file() or path.suffix.lower() not in CHECKPOINT_SUFFIXES:\n            continue\n        path_text = path.as_posix().lower()\n        if require_all and not all(term in path_text for term in include_terms):\n            continue\n        if not require_all and not any(term in path_text for term in include_terms):\n            continue\n        if any(term in path_text for term in exclude_terms):\n            continue\n        candidates.append(path)\n    if not candidates:\n        return None\n    return sorted(candidates, key=lambda item: (len(item.as_posix()), item.as_posix()))[0]\n\n\ndef _find_checkpoint_by_glob(root: Path, checkpoint_glob: str | None) -> Path | None:\n    if checkpoint_glob:\n        candidates = [path for path in root.rglob(checkpoint_glob) if path.is_file()]\n    else:\n        candidates = [\n            path\n            for path in root.rglob("*")\n            if path.is_file() and path.suffix.lower() in CHECKPOINT_SUFFIXES\n        ]\n    if not candidates:\n        return None\n    return sorted(candidates, key=lambda item: (len(item.as_posix()), item.as_posix()))[0]\n\n\ndef _filename_from_url(url: str) -> str:\n    filename = url.rstrip("/").split("/")[-1]\n    return filename or "checkpoint.pt"\n\n\ndef _iter_dirs(root: Path):\n    for path in root.rglob("*"):\n        if path.is_dir():\n            yield path\n', 'src/mythos/kaggle_run.py': '"""Kaggle-oriented runner for producing /kaggle/working/submission.json."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nimport sys\n\nfrom mythos.arc import ArcValidationError, load_challenges\nfrom mythos.metrics import score_files\nfrom mythos.solvers.base import SolverError\nfrom mythos.solvers.factory import make_solver\nfrom mythos.submission import write_submission\n\nDEFAULT_KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")\nDEFAULT_KAGGLE_OUTPUT = Path("/kaggle/working/submission.json")\n\nCHALLENGE_FILES = {\n    "training": ("arc-agi_training-challenges.json", "arc-agi_training_challenges.json"),\n    "evaluation": ("arc-agi_evaluation-challenges.json", "arc-agi_evaluation_challenges.json"),\n    "test": ("arc-agi_test-challenges.json", "arc-agi_test_challenges.json"),\n}\n\nSOLUTION_FILES = {\n    "training": ("arc-agi_training-solutions.json", "arc-agi_training_solutions.json"),\n    "evaluation": ("arc-agi_evaluation-solutions.json", "arc-agi_evaluation_solutions.json"),\n}\n\n\ndef resolve_challenge_path(data_dir: str | Path, split: str) -> Path:\n    return _first_existing(Path(data_dir), CHALLENGE_FILES[split])\n\n\ndef resolve_solution_path(data_dir: str | Path, split: str) -> Path | None:\n    candidates = SOLUTION_FILES.get(split)\n    if not candidates:\n        return None\n    try:\n        return _first_existing(Path(data_dir), candidates)\n    except FileNotFoundError:\n        return None\n\n\ndef _first_existing(data_dir: Path, names: tuple[str, ...]) -> Path:\n    for name in names:\n        path = data_dir / name\n        if path.exists():\n            return path\n    joined = ", ".join(names)\n    raise FileNotFoundError(f"none of these files exist in {data_dir}: {joined}")\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run Mythos against Kaggle ARC-AGI data.")\n    parser.add_argument("--data-dir", default=str(DEFAULT_KAGGLE_DATA_DIR))\n    parser.add_argument("--split", choices=sorted(CHALLENGE_FILES), default="test")\n    parser.add_argument("--challenges", help="Explicit challenge JSON path; overrides --data-dir/--split.")\n    parser.add_argument("--solutions", help="Explicit solutions JSON path for local scoring.")\n    parser.add_argument("--solver", choices=["pipeline", "baseline", "fixture", "hrm"], default="pipeline")\n    parser.add_argument("--model-mode", choices=["fallback", "strict"], default=None)\n    parser.add_argument("--out", default=str(DEFAULT_KAGGLE_OUTPUT))\n    parser.add_argument("--score", action="store_true", help="Score output when solutions are available.")\n    args = parser.parse_args(argv)\n\n    try:\n        challenge_path = Path(args.challenges) if args.challenges else resolve_challenge_path(args.data_dir, args.split)\n        solution_path = Path(args.solutions) if args.solutions else resolve_solution_path(args.data_dir, args.split)\n\n        tasks = load_challenges(challenge_path)\n        solver = make_solver(args.solver, model_mode=args.model_mode)\n        predictions = [solver.solve(task) for task in tasks.values()]\n        write_submission(predictions, args.out)\n\n        summary: dict[str, object] = {\n            "challenge_path": str(challenge_path),\n            "output_path": str(args.out),\n            "solver": args.solver,\n            "model_mode": args.model_mode or "fallback",\n            "tasks": len(tasks),\n        }\n        if hasattr(solver, "pipeline"):\n            summary["models"] = solver.pipeline.model_registry.summary()\n        if args.score and solution_path is not None:\n            summary["score"] = score_files(args.out, str(solution_path)).to_dict()\n        elif args.score:\n            summary["score"] = "skipped: no solutions file for this split"\n    except (ArcValidationError, SolverError, FileNotFoundError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(summary, indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/metrics.py': '"""Scoring helpers for ARC-style submissions."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Mapping, Tuple\n\nfrom mythos.arc import ArcValidationError, Grid, SolutionMap, grid_equal, load_solutions\nfrom mythos.submission import SubmissionMap, TestPrediction, load_submission\n\n\n@dataclass(frozen=True)\nclass ScoreResult:\n    total_items: int\n    exact_matches: int\n    exact_attempt_1: int\n    exact_attempt_2: int\n    total_cells: int\n    matched_cells: int\n    extra_predictions: int\n\n    @property\n    def exact_accuracy(self) -> float:\n        return self.exact_matches / self.total_items if self.total_items else 0.0\n\n    @property\n    def cell_accuracy(self) -> float:\n        return self.matched_cells / self.total_cells if self.total_cells else 0.0\n\n    def to_dict(self) -> dict[str, int | float]:\n        data = asdict(self)\n        data["exact_accuracy"] = self.exact_accuracy\n        data["cell_accuracy"] = self.cell_accuracy\n        return data\n\n\ndef _cell_count(grid: Grid) -> int:\n    return sum(len(row) for row in grid)\n\n\ndef _cell_matches(prediction: Grid, truth: Grid) -> Tuple[int, int]:\n    total = _cell_count(truth)\n    if len(prediction) != len(truth) or len(prediction[0]) != len(truth[0]):\n        return 0, total\n    matches = 0\n    for pred_row, truth_row in zip(prediction, truth):\n        for pred_cell, truth_cell in zip(pred_row, truth_row):\n            if pred_cell == truth_cell:\n                matches += 1\n    return matches, total\n\n\ndef score_submission_data(predictions: SubmissionMap, solutions: SolutionMap) -> ScoreResult:\n    total_items = 0\n    exact_matches = 0\n    exact_attempt_1 = 0\n    exact_attempt_2 = 0\n    matched_cells = 0\n    total_cells = 0\n\n    extra_predictions = len(set(predictions) - set(solutions))\n\n    for task_id, truth_outputs in solutions.items():\n        if task_id not in predictions:\n            raise ArcValidationError(f"submission is missing task {task_id}")\n        task_predictions = predictions[task_id]\n        if len(task_predictions) != len(truth_outputs):\n            raise ArcValidationError(\n                f"{task_id} has {len(task_predictions)} predictions but "\n                f"{len(truth_outputs)} solution outputs"\n            )\n\n        for prediction, truth in zip(task_predictions, truth_outputs):\n            total_items += 1\n            attempt_1_exact = grid_equal(prediction.attempt_1, truth)\n            attempt_2_exact = grid_equal(prediction.attempt_2, truth)\n            exact_attempt_1 += int(attempt_1_exact)\n            exact_attempt_2 += int(attempt_2_exact)\n            exact_matches += int(attempt_1_exact or attempt_2_exact)\n\n            cells_1, cells_total = _cell_matches(prediction.attempt_1, truth)\n            cells_2, _ = _cell_matches(prediction.attempt_2, truth)\n            matched_cells += max(cells_1, cells_2)\n            total_cells += cells_total\n\n    return ScoreResult(\n        total_items=total_items,\n        exact_matches=exact_matches,\n        exact_attempt_1=exact_attempt_1,\n        exact_attempt_2=exact_attempt_2,\n        total_cells=total_cells,\n        matched_cells=matched_cells,\n        extra_predictions=extra_predictions,\n    )\n\n\ndef score_files(prediction_path: str, solution_path: str) -> ScoreResult:\n    return score_submission_data(load_submission(prediction_path), load_solutions(solution_path))\n', 'src/mythos/models.py': '"""External model loading for the Project Mythos pipeline."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nimport importlib\nimport os\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nfrom mythos.solvers.base import SolverError\n\n\nclass ModelLoadError(SolverError):\n    """Raised when a configured external model cannot be loaded."""\n\n\n@dataclass(frozen=True)\nclass ModelSpec:\n    key: str\n    label: str\n    checkpoint_env: str\n    repo_env: str | None = None\n    module_names: tuple[str, ...] = ()\n\n\nMODEL_SPECS: tuple[ModelSpec, ...] = (\n    ModelSpec(\n        key="jepa",\n        label="JEPA encoder",\n        repo_env="IJEPA_REPO_DIR",\n        checkpoint_env="IJEPA_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="hrm_text",\n        label="HRM-Text H-module",\n        repo_env="HRM_TEXT_REPO_DIR",\n        checkpoint_env="HRM_TEXT_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="world_model",\n        label="World model",\n        checkpoint_env="WORLD_MODEL_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="ttt_lora",\n        label="TTT LoRA adapters",\n        checkpoint_env="TTT_LORA_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="hrm_l_module",\n        label="HRM 27M L-module",\n        repo_env="HRM_REPO_DIR",\n        checkpoint_env="HRM_CHECKPOINT_PATH",\n        module_names=("pretrain", "evaluate"),\n    ),\n)\n\n\n@dataclass(frozen=True)\nclass LoadedModel:\n    spec: ModelSpec\n    repo_dir: Path | None = None\n    checkpoint_path: Path | None = None\n    checkpoint: Any = None\n    modules: dict[str, Any] = field(default_factory=dict)\n    error: str | None = None\n\n    @property\n    def loaded(self) -> bool:\n        return self.checkpoint is not None\n\n    def describe(self) -> str:\n        if not self.loaded:\n            if self.error:\n                return f"{self.spec.label}: load failed: {self.error}"\n            return f"{self.spec.label}: not configured"\n        parts = [f"{self.spec.label}: checkpoint={self.checkpoint_path}"]\n        if self.repo_dir is not None:\n            parts.append(f"repo={self.repo_dir}")\n        if self.modules:\n            parts.append(f"modules={\',\'.join(sorted(self.modules))}")\n        return "; ".join(parts)\n\n\nclass ModelRegistry:\n    """Loads and stores external models keyed by planned pipeline component."""\n\n    def __init__(self, models: dict[str, LoadedModel], *, strict: bool) -> None:\n        self.models = models\n        self.strict = strict\n\n    @classmethod\n    def from_env(cls, *, strict: bool = False) -> "ModelRegistry":\n        models: dict[str, LoadedModel] = {}\n        for spec in MODEL_SPECS:\n            try:\n                models[spec.key] = _load_model_from_env(spec, strict=strict)\n            except ModelLoadError as exc:\n                if strict:\n                    raise\n                models[spec.key] = LoadedModel(spec=spec, error=str(exc))\n        return cls(models=models, strict=strict)\n\n    def get(self, key: str) -> LoadedModel:\n        return self.models[key]\n\n    def summary(self) -> list[dict[str, object]]:\n        return [\n            {\n                "key": key,\n                "label": model.spec.label,\n                "loaded": model.loaded,\n                "repo_dir": str(model.repo_dir) if model.repo_dir is not None else None,\n                "checkpoint_path": str(model.checkpoint_path) if model.checkpoint_path is not None else None,\n                "modules": sorted(model.modules),\n                "error": model.error,\n            }\n            for key, model in self.models.items()\n        ]\n\n\ndef _load_model_from_env(spec: ModelSpec, *, strict: bool) -> LoadedModel:\n    repo_value = os.environ.get(spec.repo_env) if spec.repo_env is not None else None\n    checkpoint_value = os.environ.get(spec.checkpoint_env)\n\n    if not repo_value and not checkpoint_value:\n        if strict:\n            missing = spec.checkpoint_env\n            if spec.repo_env is not None:\n                missing = f"{spec.repo_env} and {spec.checkpoint_env}"\n            raise ModelLoadError(f"{missing} are required for strict model loading")\n        return LoadedModel(spec=spec)\n\n    if spec.repo_env is not None and not repo_value:\n        raise ModelLoadError(f"{spec.repo_env} is required when loading {spec.label}")\n    if not checkpoint_value:\n        raise ModelLoadError(f"{spec.checkpoint_env} is required when loading {spec.label}")\n\n    repo_dir = Path(repo_value) if repo_value else None\n    checkpoint_path = Path(checkpoint_value)\n\n    if repo_dir is not None:\n        if not repo_dir.exists():\n            raise ModelLoadError(f"{spec.repo_env} does not exist: {repo_dir}")\n        _add_repo_to_path(repo_dir)\n\n    if not checkpoint_path.exists():\n        raise ModelLoadError(f"{spec.checkpoint_env} does not exist: {checkpoint_path}")\n\n    modules = _import_modules(spec)\n    checkpoint = _load_torch_checkpoint(checkpoint_path)\n    return LoadedModel(\n        spec=spec,\n        repo_dir=repo_dir,\n        checkpoint_path=checkpoint_path,\n        checkpoint=checkpoint,\n        modules=modules,\n    )\n\n\ndef _add_repo_to_path(repo_dir: Path) -> None:\n    repo = str(repo_dir.resolve())\n    if repo not in sys.path:\n        sys.path.insert(0, repo)\n\n\ndef _import_modules(spec: ModelSpec) -> dict[str, Any]:\n    modules: dict[str, Any] = {}\n    for module_name in spec.module_names:\n        try:\n            modules[module_name] = importlib.import_module(module_name)\n        except Exception as exc:\n            raise ModelLoadError(\n                f"failed to import {module_name!r} for {spec.label}: {exc}"\n            ) from exc\n    return modules\n\n\ndef _load_torch_checkpoint(checkpoint_path: Path) -> Any:\n    try:\n        torch = importlib.import_module("torch")\n    except Exception as exc:\n        raise ModelLoadError("PyTorch is required to load model checkpoints") from exc\n\n    map_location = "cuda" if torch.cuda.is_available() else "cpu"\n    try:\n        return torch.load(checkpoint_path, map_location=map_location, weights_only=False)\n    except TypeError:\n        return torch.load(checkpoint_path, map_location=map_location)\n', 'src/mythos/pipeline.py': '"""Plan-aligned Project Mythos inference pipeline.\n\nThe real research components are still adapters here. The important point for\nthe base implementation is that data flows through the same stage boundaries as\nthe master plan, so each placeholder has an obvious replacement point.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nfrom typing import Iterable, Tuple\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.models import ModelRegistry\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.submission import Prediction, prediction_to_json, validate_submission_data\n\nPLAN_STAGE_ORDER = (\n    "ingest",\n    "encode_jepa",\n    "plan_hrm_text",\n    "simulate_world_model",\n    "adapt_ttt_lora",\n    "execute_hrm_l_module",\n    "decode_output",\n)\n\n\n@dataclass(frozen=True)\nclass StageRecord:\n    name: str\n    status: str\n    detail: str\n\n\n@dataclass(frozen=True)\nclass GridEmbedding:\n    source: str\n    shape: tuple[int, int]\n    vector: tuple[float, ...]\n\n\n@dataclass(frozen=True)\nclass RuleVector:\n    source: str\n    description: str\n    vector: tuple[float, ...]\n\n\n@dataclass\nclass PipelineTrace:\n    task_id: str\n    stages: list[StageRecord] = field(default_factory=list)\n\n    @property\n    def stage_names(self) -> list[str]:\n        return [stage.name for stage in self.stages]\n\n    def add(self, name: str, status: str, detail: str) -> None:\n        self.stages.append(StageRecord(name=name, status=status, detail=detail))\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "task_id": self.task_id,\n            "stages": [\n                {"name": stage.name, "status": stage.status, "detail": stage.detail}\n                for stage in self.stages\n            ],\n        }\n\n\n@dataclass(frozen=True)\nclass PipelineResult:\n    prediction: Prediction\n    trace: PipelineTrace\n\n\n@dataclass\nclass PipelineState:\n    task: ArcTask\n    trace: PipelineTrace\n    embeddings: tuple[GridEmbedding, ...] = ()\n    rule_vector: RuleVector | None = None\n    prediction: Prediction | None = None\n\n\nclass PlannedPipeline:\n    """Runs the ARC task through the Project Mythos master-plan stages."""\n\n    def __init__(\n        self,\n        executor: BaselineSolver | None = None,\n        model_registry: ModelRegistry | None = None,\n        *,\n        strict_models: bool = False,\n    ) -> None:\n        self.executor = executor or BaselineSolver()\n        self.model_registry = model_registry or ModelRegistry.from_env(strict=strict_models)\n\n    def run(self, task: ArcTask) -> PipelineResult:\n        state = PipelineState(task=task, trace=PipelineTrace(task_id=task.id))\n\n        self._ingest(state)\n        self._encode_jepa(state)\n        self._plan_hrm_text(state)\n        self._simulate_world_model(state)\n        self._adapt_ttt_lora(state)\n        self._execute_hrm_l_module(state)\n        self._decode_output(state)\n\n        assert state.prediction is not None\n        return PipelineResult(prediction=state.prediction, trace=state.trace)\n\n    def _ingest(self, state: PipelineState) -> None:\n        train_count = len(state.task.train)\n        test_count = len(state.task.test)\n        state.trace.add(\n            "ingest",\n            "ok",\n            f"loaded task with {train_count} train pairs and {test_count} test inputs",\n        )\n\n    def _encode_jepa(self, state: PipelineState) -> None:\n        model = self.model_registry.get("jepa")\n        embeddings: list[GridEmbedding] = []\n        for split, grids in _iter_task_grids(state.task):\n            for index, grid in enumerate(grids):\n                embeddings.append(_encode_grid(f"{split}[{index}]", grid))\n        state.embeddings = tuple(embeddings)\n        status = "model_loaded" if model.loaded else "fallback"\n        detail = (\n            f"{model.describe()}; deterministic grid-feature adapter produced "\n            f"{len(embeddings)} embeddings"\n        )\n        state.trace.add(\n            "encode_jepa",\n            status,\n            detail,\n        )\n\n    def _plan_hrm_text(self, state: PipelineState) -> None:\n        model = self.model_registry.get("hrm_text")\n        state.rule_vector = _make_rule_vector(state.task)\n        status = "model_loaded" if model.loaded else "fallback"\n        state.trace.add(\n            "plan_hrm_text",\n            status,\n            f"{model.describe()}; rule vector derived from train-pair deltas",\n        )\n\n    def _simulate_world_model(self, state: PipelineState) -> None:\n        if state.rule_vector is None:\n            raise RuntimeError("rule vector must exist before world-model simulation")\n        model = self.model_registry.get("world_model")\n        status = "model_loaded" if model.loaded else "fallback"\n        state.trace.add(\n            "simulate_world_model",\n            status,\n            f"{model.describe()}; rollout hook passed rule vector through",\n        )\n\n    def _adapt_ttt_lora(self, state: PipelineState) -> None:\n        model = self.model_registry.get("ttt_lora")\n        status = "model_loaded" if model.loaded else "fallback"\n        state.trace.add(\n            "adapt_ttt_lora",\n            status,\n            f"{model.describe()}; per-task adapter update hook completed",\n        )\n\n    def _execute_hrm_l_module(self, state: PipelineState) -> None:\n        model = self.model_registry.get("hrm_l_module")\n        state.prediction = self.executor.solve(state.task)\n        status = "model_loaded_fallback_executor" if model.loaded else "fallback"\n        detail = (\n            f"{model.describe()}; baseline executor produced valid output because "\n            "direct HRM inference is not implemented in this adapter yet"\n        )\n        state.trace.add(\n            "execute_hrm_l_module",\n            status,\n            detail,\n        )\n\n    def _decode_output(self, state: PipelineState) -> None:\n        if state.prediction is None:\n            raise RuntimeError("prediction must exist before decode/output")\n        validate_submission_data({state.prediction.task_id: prediction_to_json(state.prediction)})\n        state.trace.add(\n            "decode_output",\n            "ok",\n            f"validated {len(state.prediction.outputs)} two-attempt test predictions",\n        )\n\n\ndef _iter_task_grids(task: ArcTask) -> Iterable[tuple[str, tuple[Grid, ...]]]:\n    yield "train_input", tuple(example.input for example in task.train)\n    yield "train_output", tuple(example.output for example in task.train if example.output is not None)\n    yield "test_input", tuple(example.input for example in task.test)\n\n\ndef _encode_grid(source: str, grid: Grid) -> GridEmbedding:\n    height = len(grid)\n    width = len(grid[0])\n    flat = [cell for row in grid for cell in row]\n    counts = Counter(flat)\n    dominant_color = counts.most_common(1)[0][0]\n    nonzero = sum(1 for cell in flat if cell != 0)\n    total = len(flat)\n    vector = (\n        height / 30.0,\n        width / 30.0,\n        dominant_color / 9.0,\n        nonzero / total,\n        sum(flat) / (9.0 * total),\n    )\n    return GridEmbedding(source=source, shape=(height, width), vector=_rounded(vector))\n\n\ndef _make_rule_vector(task: ArcTask) -> RuleVector:\n    shape_deltas: list[tuple[int, int]] = []\n    nonzero_deltas: list[int] = []\n    color_jaccards: list[float] = []\n\n    for example in task.train:\n        if example.output is None:\n            continue\n        in_h, in_w = len(example.input), len(example.input[0])\n        out_h, out_w = len(example.output), len(example.output[0])\n        shape_deltas.append((out_h - in_h, out_w - in_w))\n        nonzero_deltas.append(_nonzero_count(example.output) - _nonzero_count(example.input))\n        color_jaccards.append(_color_jaccard(example.input, example.output))\n\n    avg_shape_delta_h = _average(delta[0] for delta in shape_deltas)\n    avg_shape_delta_w = _average(delta[1] for delta in shape_deltas)\n    avg_nonzero_delta = _average(nonzero_deltas)\n    avg_color_overlap = _average(color_jaccards)\n    vector = _rounded(\n        (\n            avg_shape_delta_h / 30.0,\n            avg_shape_delta_w / 30.0,\n            avg_nonzero_delta / 900.0,\n            avg_color_overlap,\n        )\n    )\n    return RuleVector(\n        source="deterministic_train_pair_delta",\n        description="shape, density, and color-overlap summary from demonstration pairs",\n        vector=vector,\n    )\n\n\ndef _nonzero_count(grid: Grid) -> int:\n    return sum(1 for row in grid for cell in row if cell != 0)\n\n\ndef _color_jaccard(left: Grid, right: Grid) -> float:\n    left_colors = {cell for row in left for cell in row}\n    right_colors = {cell for row in right for cell in row}\n    union = left_colors | right_colors\n    if not union:\n        return 1.0\n    return len(left_colors & right_colors) / len(union)\n\n\ndef _average(values: Iterable[float]) -> float:\n    collected = list(values)\n    return sum(collected) / len(collected) if collected else 0.0\n\n\ndef _rounded(values: Iterable[float]) -> Tuple[float, ...]:\n    return tuple(round(value, 6) for value in values)\n', 'src/mythos/score.py': '"""CLI for scoring a submission against solution JSON."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.arc import ArcValidationError\nfrom mythos.metrics import score_files\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Score an ARC submission JSON file.")\n    parser.add_argument("--pred", required=True, help="Path to submission JSON.")\n    parser.add_argument("--solutions", required=True, help="Path to solution JSON.")\n    args = parser.parse_args(argv)\n\n    try:\n        result = score_files(args.pred, args.solutions)\n    except ArcValidationError as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/solve.py': '"""CLI for running a solver and writing submission JSON."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nfrom mythos.arc import ArcValidationError, load_challenges\nfrom mythos.solvers.base import SolverError\nfrom mythos.solvers.factory import make_solver\nfrom mythos.submission import write_submission\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run a Mythos solver.")\n    parser.add_argument("--solver", choices=["pipeline", "baseline", "fixture", "hrm"], default="pipeline")\n    parser.add_argument("--model-mode", choices=["fallback", "strict"], default=None)\n    parser.add_argument("--challenges", required=True, help="Path to ARC-style challenges JSON.")\n    parser.add_argument("--out", required=True, help="Output submission JSON path.")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.challenges)\n        solver = make_solver(args.solver, model_mode=args.model_mode)\n        predictions = [solver.solve(task) for task in tasks.values()]\n        write_submission(predictions, args.out)\n    except (ArcValidationError, SolverError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    print(f"Wrote {len(predictions)} predictions to {args.out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/solvers/__init__.py': '"""Solver implementations."""\n\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.base import Solver, SolverError\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.solvers.hrm import HRMEnvironmentError, HRMSolver\nfrom mythos.solvers.pipeline import PlannedPipelineSolver\n\n__all__ = [\n    "BaselineSolver",\n    "FixtureSolver",\n    "HRMEnvironmentError",\n    "HRMSolver",\n    "PlannedPipelineSolver",\n    "Solver",\n    "SolverError",\n]\n', 'src/mythos/solvers/base.py': '"""Common solver types."""\n\nfrom __future__ import annotations\n\nfrom typing import Protocol\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.submission import Prediction, TestPrediction\n\n\nclass SolverError(RuntimeError):\n    """Raised when a solver cannot produce a prediction."""\n\n\nclass Solver(Protocol):\n    def solve(self, task: ArcTask) -> Prediction:\n        """Return a two-attempt prediction for every test item in a task."""\n\n\ndef make_prediction(task: ArcTask, attempts: list[tuple[Grid, Grid]]) -> Prediction:\n    if len(attempts) != len(task.test):\n        raise SolverError(\n            f"{task.id}: expected {len(task.test)} test predictions, got {len(attempts)}"\n        )\n    return Prediction(\n        task_id=task.id,\n        outputs=tuple(TestPrediction(attempt_1=a1, attempt_2=a2) for a1, a2 in attempts),\n    )\n', 'src/mythos/solvers/baseline.py': '"""Guaranteed-output baseline solver for smoke runs and Kaggle plumbing tests."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\n\nfrom mythos.arc import ArcTask, Grid, copy_grid\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.submission import Prediction\n\n\nclass BaselineSolver:\n    """Try simple fixture rules, then fall back to valid low-skill predictions."""\n\n    def __init__(self) -> None:\n        self.fixture_solver = FixtureSolver()\n\n    def solve(self, task: ArcTask) -> Prediction:\n        try:\n            return self.fixture_solver.solve(task)\n        except SolverError:\n            attempts = [\n                (copy_grid(example.input), _blank_output_for_task(task, example.input))\n                for example in task.test\n            ]\n            return make_prediction(task, attempts)\n\n\ndef _blank_output_for_task(task: ArcTask, input_grid: Grid) -> Grid:\n    height, width = _fallback_shape(task, input_grid)\n    color = _dominant_output_color(task)\n    return [[color for _ in range(width)] for _ in range(height)]\n\n\ndef _fallback_shape(task: ArcTask, input_grid: Grid) -> tuple[int, int]:\n    output_shapes = {\n        (len(example.output), len(example.output[0]))\n        for example in task.train\n        if example.output is not None\n    }\n    if len(output_shapes) == 1:\n        return next(iter(output_shapes))\n    return len(input_grid), len(input_grid[0])\n\n\ndef _dominant_output_color(task: ArcTask) -> int:\n    counts: Counter[int] = Counter()\n    for example in task.train:\n        if example.output is None:\n            continue\n        for row in example.output:\n            counts.update(row)\n    if not counts:\n        return 0\n    return counts.most_common(1)[0][0]\n', 'src/mythos/solvers/factory.py': '"""Solver factory shared by CLIs."""\n\nfrom __future__ import annotations\n\nimport os\n\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.solvers.hrm import HRMSolver\nfrom mythos.solvers.pipeline import PlannedPipelineSolver\n\n\ndef make_solver(name: str, *, model_mode: str | None = None):\n    selected_mode = model_mode or os.environ.get("MYTHOS_MODEL_MODE", "fallback")\n    if selected_mode not in {"fallback", "strict"}:\n        raise ValueError(f"unknown model mode: {selected_mode}")\n    strict_models = selected_mode == "strict"\n    if name == "pipeline":\n        return PlannedPipelineSolver(strict_models=strict_models)\n    if name == "baseline":\n        return BaselineSolver()\n    if name == "fixture":\n        return FixtureSolver()\n    if name == "hrm":\n        return HRMSolver()\n    raise ValueError(f"unknown solver: {name}")\n', 'src/mythos/solvers/fixture.py': '"""Small deterministic solver for the committed toy fixtures."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Callable, Iterable, List, Optional, Tuple\n\nfrom mythos.arc import ArcTask, Grid, copy_grid, grid_equal\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.submission import Prediction\n\nTransform = Callable[[Grid], Grid]\n\n\n@dataclass(frozen=True)\nclass Candidate:\n    name: str\n    transform: Transform\n\n\nclass FixtureSolver:\n    """Infer one simple transformation from train examples and apply it to tests."""\n\n    def solve(self, task: ArcTask) -> Prediction:\n        candidates = _matching_candidates(task)\n        if not candidates:\n            raise SolverError(f"{task.id}: no fixture transformation matched train examples")\n\n        primary = candidates[0].transform\n        secondary = candidates[1].transform if len(candidates) > 1 else primary\n        attempts = [(primary(example.input), secondary(example.input)) for example in task.test]\n        return make_prediction(task, attempts)\n\n\ndef _matching_candidates(task: ArcTask) -> List[Candidate]:\n    candidates = _base_candidates()\n    candidates.extend(_translation_candidates(task))\n    recolor = _recolor_candidate(task)\n    if recolor is not None:\n        candidates.append(recolor)\n\n    matched: List[Candidate] = []\n    seen_outputs: set[str] = set()\n    for candidate in candidates:\n        if _fits(task, candidate.transform):\n            signature = _candidate_signature(task, candidate.transform)\n            if signature not in seen_outputs:\n                matched.append(candidate)\n                seen_outputs.add(signature)\n    return matched\n\n\ndef _candidate_signature(task: ArcTask, transform: Transform) -> str:\n    return repr([transform(example.input) for example in task.test])\n\n\ndef _fits(task: ArcTask, transform: Transform) -> bool:\n    for example in task.train:\n        if example.output is None:\n            return False\n        try:\n            predicted = transform(example.input)\n        except ValueError:\n            return False\n        if not grid_equal(predicted, example.output):\n            return False\n    return True\n\n\ndef _base_candidates() -> List[Candidate]:\n    return [\n        Candidate("identity", copy_grid),\n        Candidate("mirror_horizontal", _mirror_horizontal),\n        Candidate("mirror_vertical", _mirror_vertical),\n        Candidate("rotate_clockwise", _rotate_clockwise),\n        Candidate("rotate_180", lambda grid: _rotate_clockwise(_rotate_clockwise(grid))),\n        Candidate("rotate_counterclockwise", _rotate_counterclockwise),\n    ]\n\n\ndef _mirror_horizontal(grid: Grid) -> Grid:\n    return [list(reversed(row)) for row in grid]\n\n\ndef _mirror_vertical(grid: Grid) -> Grid:\n    return [row[:] for row in reversed(grid)]\n\n\ndef _rotate_clockwise(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid[::-1])]\n\n\ndef _rotate_counterclockwise(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid)][::-1]\n\n\ndef _recolor_candidate(task: ArcTask) -> Optional[Candidate]:\n    mapping: dict[int, int] = {}\n    for example in task.train:\n        if example.output is None:\n            return None\n        if len(example.input) != len(example.output) or len(example.input[0]) != len(example.output[0]):\n            return None\n        for in_row, out_row in zip(example.input, example.output):\n            for in_cell, out_cell in zip(in_row, out_row):\n                previous = mapping.setdefault(in_cell, out_cell)\n                if previous != out_cell:\n                    return None\n\n    def transform(grid: Grid) -> Grid:\n        return [[mapping.get(cell, cell) for cell in row] for row in grid]\n\n    return Candidate("recolor", transform)\n\n\ndef _translation_candidates(task: ArcTask) -> List[Candidate]:\n    offsets: Optional[set[tuple[int, int]]] = None\n    for example in task.train:\n        if example.output is None:\n            return []\n        example_offsets = set(_valid_translation_offsets(example.input, example.output))\n        offsets = example_offsets if offsets is None else offsets & example_offsets\n    return [\n        Candidate(f"translate_{dr}_{dc}", _translate_transform(dr, dc))\n        for dr, dc in sorted(offsets or set())\n        if dr != 0 or dc != 0\n    ]\n\n\ndef _valid_translation_offsets(source: Grid, target: Grid) -> Iterable[tuple[int, int]]:\n    if len(source) != len(target) or len(source[0]) != len(target[0]):\n        return []\n\n    source_cells = _foreground_cells(source)\n    target_cells = _foreground_cells(target)\n    if len(source_cells) != len(target_cells):\n        return []\n    if not source_cells and not target_cells:\n        return [(0, 0)]\n\n    offsets = []\n    first_r, first_c, first_value = source_cells[0]\n    for target_r, target_c, target_value in target_cells:\n        if target_value != first_value:\n            continue\n        dr = target_r - first_r\n        dc = target_c - first_c\n        try:\n            translated = _translate_grid(source, dr, dc)\n        except ValueError:\n            continue\n        if translated == target:\n            offsets.append((dr, dc))\n    return offsets\n\n\ndef _foreground_cells(grid: Grid) -> List[tuple[int, int, int]]:\n    return [\n        (row_idx, col_idx, cell)\n        for row_idx, row in enumerate(grid)\n        for col_idx, cell in enumerate(row)\n        if cell != 0\n    ]\n\n\ndef _translate_transform(dr: int, dc: int) -> Transform:\n    def transform(grid: Grid) -> Grid:\n        return _translate_grid(grid, dr, dc)\n\n    return transform\n\n\ndef _translate_grid(grid: Grid, dr: int, dc: int) -> Grid:\n    height = len(grid)\n    width = len(grid[0])\n    translated = [[0 for _ in range(width)] for _ in range(height)]\n    for row_idx, row in enumerate(grid):\n        for col_idx, cell in enumerate(row):\n            if cell == 0:\n                continue\n            next_r = row_idx + dr\n            next_c = col_idx + dc\n            if next_r < 0 or next_r >= height or next_c < 0 or next_c >= width:\n                raise ValueError("translation moves cell out of bounds")\n            translated[next_r][next_c] = cell\n    return translated\n', 'src/mythos/solvers/hrm.py': '"""External HRM adapter and environment checks."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport importlib\nimport os\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nfrom mythos.arc import ArcTask\nfrom mythos.solvers.base import SolverError\nfrom mythos.submission import Prediction\n\n\nclass HRMEnvironmentError(SolverError):\n    """Raised when the external HRM runtime is not ready."""\n\n\n@dataclass(frozen=True)\nclass HRMEnvironment:\n    repo_dir: Path\n    checkpoint_path: Path\n\n    @classmethod\n    def from_env(cls) -> "HRMEnvironment":\n        repo_value = os.environ.get("HRM_REPO_DIR")\n        checkpoint_value = os.environ.get("HRM_CHECKPOINT_PATH")\n        if not repo_value:\n            raise HRMEnvironmentError("HRM_REPO_DIR is required for HRM execution")\n        if not checkpoint_value:\n            raise HRMEnvironmentError("HRM_CHECKPOINT_PATH is required for HRM execution")\n        return cls(repo_dir=Path(repo_value), checkpoint_path=Path(checkpoint_value))\n\n    def validate(self, *, require_cuda: bool = True) -> None:\n        if not self.repo_dir.exists():\n            raise HRMEnvironmentError(f"HRM_REPO_DIR does not exist: {self.repo_dir}")\n        if not (self.repo_dir / "evaluate.py").exists():\n            raise HRMEnvironmentError(f"HRM checkout is missing evaluate.py: {self.repo_dir}")\n        if not (self.repo_dir / "dataset" / "build_arc_dataset.py").exists():\n            raise HRMEnvironmentError(\n                f"HRM checkout is missing dataset/build_arc_dataset.py: {self.repo_dir}"\n            )\n        if not self.checkpoint_path.exists():\n            raise HRMEnvironmentError(f"HRM_CHECKPOINT_PATH does not exist: {self.checkpoint_path}")\n\n        torch = self._import_torch()\n        if require_cuda and not torch.cuda.is_available():\n            raise HRMEnvironmentError("HRM execution requires CUDA; torch.cuda.is_available() is false")\n\n    def import_modules(self) -> dict[str, Any]:\n        self._add_repo_to_path()\n        modules = {}\n        for module_name in ("pretrain", "evaluate"):\n            try:\n                modules[module_name] = importlib.import_module(module_name)\n            except Exception as exc:  # pragma: no cover - depends on external HRM deps.\n                raise HRMEnvironmentError(\n                    f"failed to import HRM module {module_name!r} from {self.repo_dir}: {exc}"\n                ) from exc\n        return modules\n\n    def load_checkpoint(self) -> Any:\n        torch = self._import_torch()\n        map_location = "cuda" if torch.cuda.is_available() else "cpu"\n        try:\n            return torch.load(\n                self.checkpoint_path,\n                map_location=map_location,\n                weights_only=False,\n            )\n        except TypeError:\n            try:\n                return torch.load(self.checkpoint_path, map_location=map_location)\n            except Exception as exc:  # pragma: no cover - depends on checkpoint format.\n                raise HRMEnvironmentError(\n                    f"failed to load HRM checkpoint {self.checkpoint_path}: {exc}"\n                ) from exc\n        except Exception as exc:  # pragma: no cover - depends on checkpoint format.\n            raise HRMEnvironmentError(f"failed to load HRM checkpoint {self.checkpoint_path}: {exc}") from exc\n\n    def _add_repo_to_path(self) -> None:\n        repo = str(self.repo_dir.resolve())\n        if repo not in sys.path:\n            sys.path.insert(0, repo)\n\n    @staticmethod\n    def _import_torch() -> Any:\n        try:\n            return importlib.import_module("torch")\n        except Exception as exc:  # pragma: no cover - torch is optional locally.\n            raise HRMEnvironmentError("PyTorch is required for HRM execution") from exc\n\n\nclass HRMSolver:\n    """Placeholder real-model solver that fails early unless HRM is configured."""\n\n    def __init__(self, env: HRMEnvironment | None = None) -> None:\n        self.env = env\n\n    def solve(self, task: ArcTask) -> Prediction:\n        env = self.env or HRMEnvironment.from_env()\n        env.validate(require_cuda=True)\n        raise HRMEnvironmentError(\n            f"{task.id}: HRM environment is valid, but direct prediction wiring is not implemented yet. "\n            "Use `python -m mythos.hrm_smoke` to validate the external runtime and dataset path."\n        )\n', 'src/mythos/solvers/pipeline.py': '"""Solver wrapper for the plan-aligned Project Mythos pipeline."""\n\nfrom __future__ import annotations\n\nfrom mythos.arc import ArcTask\nfrom mythos.models import ModelRegistry\nfrom mythos.pipeline import PipelineTrace, PlannedPipeline\nfrom mythos.submission import Prediction\n\n\nclass PlannedPipelineSolver:\n    """Solver that runs every task through the master-plan stage boundaries."""\n\n    def __init__(\n        self,\n        pipeline: PlannedPipeline | None = None,\n        *,\n        model_registry: ModelRegistry | None = None,\n        strict_models: bool = False,\n    ) -> None:\n        self.pipeline = pipeline or PlannedPipeline(\n            model_registry=model_registry,\n            strict_models=strict_models,\n        )\n        self.traces: dict[str, PipelineTrace] = {}\n        self.last_trace: PipelineTrace | None = None\n\n    def solve(self, task: ArcTask) -> Prediction:\n        result = self.pipeline.run(task)\n        self.traces[task.id] = result.trace\n        self.last_trace = result.trace\n        return result.prediction\n', 'src/mythos/submission.py': '"""Prediction and submission JSON helpers."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Tuple\n\nfrom mythos.arc import ArcValidationError, Grid, validate_grid\n\n\n@dataclass(frozen=True)\nclass TestPrediction:\n    __test__ = False\n\n    attempt_1: Grid\n    attempt_2: Grid\n\n\n@dataclass(frozen=True)\nclass Prediction:\n    task_id: str\n    outputs: Tuple[TestPrediction, ...]\n\n\nSubmissionMap = Dict[str, Tuple[TestPrediction, ...]]\n\n\ndef prediction_to_json(prediction: Prediction) -> List[dict[str, Grid]]:\n    return [\n        {"attempt_1": item.attempt_1, "attempt_2": item.attempt_2}\n        for item in prediction.outputs\n    ]\n\n\ndef predictions_to_submission(predictions: Iterable[Prediction]) -> dict[str, List[dict[str, Grid]]]:\n    submission: dict[str, List[dict[str, Grid]]] = {}\n    for prediction in predictions:\n        if prediction.task_id in submission:\n            raise ArcValidationError(f"duplicate prediction for task {prediction.task_id}")\n        submission[prediction.task_id] = prediction_to_json(prediction)\n    if not submission:\n        raise ArcValidationError("submission must contain at least one prediction")\n    return submission\n\n\ndef validate_submission_data(data: Any) -> SubmissionMap:\n    if not isinstance(data, Mapping) or not data:\n        raise ArcValidationError("submission must be a non-empty object keyed by task id")\n    validated: SubmissionMap = {}\n    for task_id, raw_outputs in data.items():\n        if not isinstance(raw_outputs, list) or not raw_outputs:\n            raise ArcValidationError(f"{task_id} must contain a non-empty list of test outputs")\n        outputs: List[TestPrediction] = []\n        for index, raw_output in enumerate(raw_outputs):\n            if not isinstance(raw_output, Mapping):\n                raise ArcValidationError(f"{task_id}[{index}] must be an object")\n            if "attempt_1" not in raw_output or "attempt_2" not in raw_output:\n                raise ArcValidationError(f"{task_id}[{index}] must contain attempt_1 and attempt_2")\n            outputs.append(\n                TestPrediction(\n                    attempt_1=validate_grid(raw_output["attempt_1"], field=f"{task_id}[{index}].attempt_1"),\n                    attempt_2=validate_grid(raw_output["attempt_2"], field=f"{task_id}[{index}].attempt_2"),\n                )\n            )\n        validated[str(task_id)] = tuple(outputs)\n    return validated\n\n\ndef write_submission(predictions: Iterable[Prediction], path: str | Path) -> None:\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    data = predictions_to_submission(predictions)\n    with output_path.open("w", encoding="utf-8") as handle:\n        json.dump(data, handle, indent=2)\n        handle.write("\\n")\n\n\ndef load_submission(path: str | Path) -> SubmissionMap:\n    input_path = Path(path)\n    try:\n        with input_path.open("r", encoding="utf-8") as handle:\n            raw = json.load(handle)\n    except json.JSONDecodeError as exc:\n        raise ArcValidationError(f"{input_path} is not valid JSON: {exc}") from exc\n    return validate_submission_data(raw)\n', 'src/mythos/validate.py': '"""CLI for ARC challenge validation."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nfrom mythos.arc import ArcValidationError, load_challenges\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Validate an ARC challenges.json file.")\n    parser.add_argument("path", help="Path to ARC-style challenges JSON.")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.path)\n    except ArcValidationError as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    train_count = sum(len(task.train) for task in tasks.values())\n    test_count = sum(len(task.test) for task in tasks.values())\n    print(f"OK: {len(tasks)} tasks, {train_count} train examples, {test_count} test items")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}

EMBED_ROOT = Path(os.environ.get('MYTHOS_EMBED_ROOT', '/kaggle/working/project_mythos_embedded'))
if not EMBED_ROOT.parent.exists():
    EMBED_ROOT = Path.cwd() / 'project_mythos_embedded'

for relative_path, content in EMBEDDED_FILES.items():
    path = EMBED_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

SRC_DIR = EMBED_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Embedded Mythos package written to:', SRC_DIR)
print('Embedded files:', len(EMBEDDED_FILES))


## 2. Configuration

In [ ]:
from pathlib import Path
import json
import os
import time

DATA_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
SPLIT = 'test'  # 'training', 'evaluation', or 'test'
SOLVER_NAME = 'pipeline'  # 'pipeline', 'baseline', 'fixture', or 'hrm'
MODEL_MODE = 'fallback'  # 'fallback' or 'strict'
AUTO_DOWNLOAD_GIT_CODE = True
AUTO_DOWNLOAD_HF_MODELS = True
AUTO_DOWNLOAD_DIRECT_CHECKPOINTS = True
AUTO_DISCOVER_MODELS = True
OUTPUT_PATH = Path('/kaggle/working/submission.json')
RUN_HRM_SMOKE = False

# Verified public defaults looked up from official sources.
os.environ.setdefault('HRM_GIT_REPO_URL', 'https://github.com/sapientinc/HRM.git')
os.environ.setdefault('HRM_HF_REPO_ID', 'sapientinc/HRM-checkpoint-ARC-2')
os.environ.setdefault('HRM_HF_CHECKPOINT_GLOB', '*.pt')
os.environ.setdefault('IJEPA_GIT_REPO_URL', 'https://github.com/facebookresearch/ijepa.git')
os.environ.setdefault('IJEPA_CHECKPOINT_URL', 'https://dl.fbaipublicfiles.com/ijepa/IN1K-vit.h.14-300e.pth.tar')

# Optional real-model inputs. Set these if you have additional public/private model repos.
# os.environ['IJEPA_HF_REPO_ID'] = '<org-or-user>/<ijepa-model-repo>'
# os.environ['IJEPA_HF_CHECKPOINT_GLOB'] = '*.pt'
# os.environ['HRM_TEXT_HF_REPO_ID'] = '<org-or-user>/<hrm-text-model-repo>'
# os.environ['WORLD_MODEL_HF_REPO_ID'] = '<org-or-user>/<world-model-repo>'
# os.environ['TTT_LORA_HF_REPO_ID'] = '<org-or-user>/<lora-repo>'
# os.environ['HRM_HF_REPO_ID'] = '<org-or-user>/<hrm-model-repo>'
# os.environ['HRM_HF_CHECKPOINT_GLOB'] = '*.pt'

# Or set explicit Kaggle input paths when internet/download is unavailable.
# os.environ['IJEPA_REPO_DIR'] = '/kaggle/input/<ijepa-code>/ijepa'
# os.environ['IJEPA_CHECKPOINT_PATH'] = '/kaggle/input/<jepa-checkpoint>/checkpoint.pt'
# os.environ['HRM_TEXT_REPO_DIR'] = '/kaggle/input/<hrm-text-code>/hrm-text'
# os.environ['HRM_TEXT_CHECKPOINT_PATH'] = '/kaggle/input/<hrm-text-checkpoint>/checkpoint.pt'
# os.environ['WORLD_MODEL_CHECKPOINT_PATH'] = '/kaggle/input/<world-model>/world_model.pt'
# os.environ['TTT_LORA_CHECKPOINT_PATH'] = '/kaggle/input/<lora>/lora.pt'
# os.environ['HRM_REPO_DIR'] = '/kaggle/input/<hrm-code>/HRM'
# os.environ['HRM_CHECKPOINT_PATH'] = '/kaggle/input/<hrm-checkpoint>/checkpoint.pt'

print('DATA_DIR =', DATA_DIR)
print('SPLIT =', SPLIT)
print('SOLVER_NAME =', SOLVER_NAME)
print('MODEL_MODE =', MODEL_MODE)
print('AUTO_DOWNLOAD_GIT_CODE =', AUTO_DOWNLOAD_GIT_CODE)
print('AUTO_DOWNLOAD_HF_MODELS =', AUTO_DOWNLOAD_HF_MODELS)
print('AUTO_DOWNLOAD_DIRECT_CHECKPOINTS =', AUTO_DOWNLOAD_DIRECT_CHECKPOINTS)
print('AUTO_DISCOVER_MODELS =', AUTO_DISCOVER_MODELS)
print('OUTPUT_PATH =', OUTPUT_PATH)


## 3. Import Mythos Runtime

In [ ]:
import mythos
from mythos.arc import load_challenges
from mythos.kaggle_run import resolve_challenge_path, resolve_solution_path
from mythos.kaggle_models import (
    autodiscover_model_inputs,
    download_direct_checkpoint_inputs,
    download_git_code_repositories,
    download_huggingface_model_inputs,
)
from mythos.metrics import score_files
from mythos.pipeline import PLAN_STAGE_ORDER
from mythos.solvers.factory import make_solver
from mythos.submission import load_submission, write_submission

print('Imported mythos from:', mythos.__file__)
print('PLAN_STAGE_ORDER =', ' -> '.join(PLAN_STAGE_ORDER))


## 4. Load ARC Data

In [ ]:
challenge_path = resolve_challenge_path(DATA_DIR, SPLIT)
solution_path = resolve_solution_path(DATA_DIR, SPLIT)
tasks = load_challenges(challenge_path)

train_examples = sum(len(task.train) for task in tasks.values())
test_items = sum(len(task.test) for task in tasks.values())

print('challenge_path =', challenge_path)
print('solution_path =', solution_path)
print('tasks =', len(tasks))
print('train_examples =', train_examples)
print('test_items =', test_items)


## 5. Load Models / Solver

In [ ]:
if AUTO_DOWNLOAD_GIT_CODE:
    git_download = download_git_code_repositories(apply=True)
    print('git_code_download =')
    print(json.dumps(git_download, indent=2))

if AUTO_DOWNLOAD_HF_MODELS:
    hf_download = download_huggingface_model_inputs(apply=True)
    print('huggingface_model_download =')
    print(json.dumps(hf_download, indent=2))

if AUTO_DOWNLOAD_DIRECT_CHECKPOINTS:
    direct_download = download_direct_checkpoint_inputs(apply=True)
    print('direct_checkpoint_download =')
    print(json.dumps(direct_download, indent=2))

if AUTO_DISCOVER_MODELS:
    discovery = autodiscover_model_inputs(apply=True)
    print('model_autodiscovery =')
    print(json.dumps(discovery, indent=2))

solver = make_solver(SOLVER_NAME, model_mode=MODEL_MODE)
print('Loaded solver:', solver.__class__.__name__)

if hasattr(solver, 'pipeline'):
    print('model_registry =')
    print(json.dumps(solver.pipeline.model_registry.summary(), indent=2))


## 6. Run Plan-Aligned Pipeline

In [ ]:
started = time.perf_counter()
predictions = []

for index, task in enumerate(tasks.values(), start=1):
    prediction = solver.solve(task)
    predictions.append(prediction)
    if index <= 3 or index == len(tasks):
        print(f'{index}/{len(tasks)} solved: {task.id}')

write_submission(predictions, OUTPUT_PATH)
elapsed = time.perf_counter() - started

print('Wrote submission:', OUTPUT_PATH)
print('tasks_predicted =', len(predictions))
print('elapsed_seconds =', round(elapsed, 3))

if hasattr(solver, 'last_trace') and solver.last_trace is not None:
    print('last_pipeline_trace =')
    print(json.dumps(solver.last_trace.to_dict(), indent=2))


## 7. Validate Submission and Score When Solutions Exist

In [ ]:
submission = load_submission(OUTPUT_PATH)
submission_items = sum(len(outputs) for outputs in submission.values())

print('submission_tasks =', len(submission))
print('submission_test_items =', submission_items)
print('sample_task_id =', next(iter(submission)))

if solution_path is not None:
    score = score_files(str(OUTPUT_PATH), str(solution_path))
    print('score =')
    print(json.dumps(score.to_dict(), indent=2, sort_keys=True))
else:
    print('No solutions file for this split; skipping score.')


## 8. Optional HRM Smoke Test

In [ ]:
if RUN_HRM_SMOKE:
    from mythos.hrm_dataset import prepare_hrm_raw_dataset
    from mythos.solvers.hrm import HRMEnvironment

    env = HRMEnvironment.from_env()
    env.validate(require_cuda=True)
    modules = env.import_modules()
    checkpoint = env.load_checkpoint()
    raw_dir = prepare_hrm_raw_dataset(tasks.values(), Path('/kaggle/working/hrm_smoke/raw/ARC-AGI-2/data'))

    print('HRM repo:', env.repo_dir)
    print('HRM checkpoint:', env.checkpoint_path)
    print('Imported modules:', sorted(modules))
    print('Checkpoint type:', type(checkpoint).__name__)
    print('Prepared HRM raw data:', raw_dir)
else:
    print('RUN_HRM_SMOKE is False; skipping HRM smoke test.')
